In [ ]:
# ==========================================
# mass_booklet_app.py
# ==========================================
import asyncio
import json
import re
import requests
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import docx
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_TAB_ALIGNMENT, WD_TAB_LEADER
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from datetime import datetime


# ==========================================
# 1. SCRAPING FUNCTIONS (USCCB & ThanhLinh)
# ==========================================

# SECTION_KEYWORDS = [
#     (("reading 1", "first reading", "reading i"), "reading1"),
#     (("responsorial psalm", "psalm"), "psalm"),
#     (("reading 2", "second reading", "reading ii"), "reading2"),
#     (("alleluia", "gospel acclamation", "verse before the gospel"), "alleluia"),
#     (("gospel",), "gospel"),
# ]

# def _looks_like_citation(text):
#     clean = text.strip()
#     return bool(
#         re.match(r'^(cf\.\s*)?[1-3]?\s*[A-Z][a-zA-Z]+\.?\s+\d+(:\d+)?([-,]\s*\d+)*[a-z]?$', clean) or
#         re.match(r'^[1-3]?\s*[A-Z][a-zA-Z]+\.?\s+[0-9a-zA-Z\s,–-]+$', clean) and len(clean) < 50
#     )

# def _parse_usccb_html(html_content):
#     eng_dict = {
#         "feast_day": "Daily Readings",
#         "reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []
#     }
#     soup = BeautifulSoup(html_content, 'html.parser')

#     if soup.title:
#         title_text = soup.title.get_text(strip=True)
#         clean_title = title_text.split('|')[0].split('-')[0].strip()
#         if clean_title and "daily readings" not in clean_title.lower():
#             eng_dict["feast_day"] = clean_title

#     if eng_dict["feast_day"] == "Daily Readings":
#         main_title = soup.select_one('.block-page-title h1, .content-header h1, h1.page-title')
#         if main_title:
#             t = main_title.get_text(strip=True)
#             if t and not t.lower().startswith("reading"):
#                 eng_dict["feast_day"] = t

#     current_section = None
#     current_text = []

#     def save_current_option():
#         if not current_section or not current_text:
#             return
#         full_text = "\n".join(current_text).strip()
#         if full_text and full_text not in eng_dict[current_section]:
#             eng_dict[current_section].append(full_text)

#     for block in soup.select('.b-verse, .content-header, .row'):
#         heading = block.find(['h3', 'h4', 'h5'])
#         address = block.find(class_='address')
#         body = block.find(class_='content-body')

#         if heading:
#             h_text = heading.get_text(strip=True).lower()
#             matched = None
#             for keys, section in SECTION_KEYWORDS:
#                 if any(k in h_text for k in keys):
#                     matched = section
#                     break
#             if matched:
#                 save_current_option()
#                 current_section = matched
#                 current_text = []
                
#                 if address:
#                     addr_text = address.get_text(strip=True)
#                     if addr_text:
#                         current_text.append(addr_text)

#         if current_section and body:
#             b_text = body.get_text(separator=' ', strip=True)
#             if b_text and b_text not in current_text:
#                 current_text.append(b_text)

#     if not any(eng_dict[s] for s in ["reading1", "psalm", "reading2", "gospel", "alleluia"]):
#         current_section = None
#         current_text = []
#         for tag in soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'div', 'a', 'span', 'strong', 'b']):
#             text = tag.get_text(separator=' ', strip=True)
#             if not text:
#                 continue
#             lower_text = text.lower().strip()

#             if any(footer_trigger in lower_text for footer_trigger in [
#                 "email terms & privacy", "united states conference of catholic bishops", 
#                 "listen podcast", "en español", "view calendar", "get daily readings",
#                 "lectionary for mass", "privacy policy", "usccb", "subscribe"
#             ]):
#                 if current_section == "gospel":
#                     save_current_option()
#                     current_section = None
#                 continue

#             is_heading_tag = tag.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']
#             if is_heading_tag and lower_text == "or":
#                 save_current_option()
#                 current_text = []
#                 continue

#             if is_heading_tag or len(text) < 40 or 'address' in tag.get('class', []):
#                 matched = None
#                 for keys, section in SECTION_KEYWORDS:
#                     if any(k in lower_text for k in keys):
#                         matched = section
#                         break
#                 if matched:
#                     save_current_option()
#                     current_section = matched
#                     current_text = []
#                     continue

#             if current_section:
#                 if tag.name in ['p', 'div', 'strong', 'b'] and not tag.find(['div', 'p']):
#                     if text not in current_text:
#                         current_text.append(text)

#     save_current_option()
#     return eng_dict

# async def scrape_usccb_async(date_str):
#     url = f"https://bible.usccb.org/bible/readings/{date_str}.cfm"
#     empty = {"feast_day": "Daily Readings", "reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []}
    
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(
#             headless=True,
#             args=["--disable-blink-features=AutomationControlled"]
#         )
#         context = await browser.new_context(
#             user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
#         )
#         page = await context.new_page()
        
#         try:
#             print(f"  -> Navigating to USCCB for {date_str}...")
#             await page.goto(url, wait_until="networkidle", timeout=25000)
#             await page.wait_for_timeout(3000)
            
#             title = await page.title()
#             if "checking connection" in title.lower() or "cloudflare" in title.lower():
#                 print("  -> Encountered connection check, waiting longer...")
#                 await page.wait_for_timeout(5000)
                
#             html_content = await page.content()
#         except Exception as e:
#             print(f"  -> Error loading USCCB page: {e}")
#             return empty
#         finally:
#             await browser.close()

#     return _parse_usccb_html(html_content)

# def scrape_thanhlinh(url):
#     headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
#     viet_dict = {"reading1": [], "psalm": [], "reading2": [], "alleluia": [], "gospel": []}
#     try:
#         response = requests.get(url, headers=headers)
#         response.raise_for_status()
#         response.encoding = 'utf-8' 
#     except requests.exceptions.RequestException:
#         return viet_dict
        
#     soup = BeautifulSoup(response.content, 'html.parser')
#     content_container = soup.find('div', class_=re.compile(r'node__content|field--name-body|content-article', re.I)) or soup.find('article') or soup
    
#     current_section = None
#     current_text = []

#     def save_current_section():
#         if current_section and current_text:
#             cleaned_text = "\n".join(current_text).strip()
#             if cleaned_text and cleaned_text not in viet_dict[current_section]:
#                 viet_dict[current_section].append(cleaned_text)

#     for tag in content_container.find_all(['p', 'h3', 'h4', 'div', 'span', 'strong']):
#         text = tag.get_text(separator=' ', strip=True)
#         if not text:
#             continue
#         lower_text = text.lower()
#         if any(w in lower_text for w in ["mục đặc biệt", "xem tất cả", "phim ảnh", "thánh ca phụng vụ", "tủ sách"]):
#             if current_section:
#                 save_current_section()
#                 current_section = None
#             continue

#         matched_section = None
#         if text.startswith("Bài Ðọc I:") or text.startswith("Bài Đọc I:") or text.startswith("Bài đọc I:"):
#             matched_section = "reading1"
#         elif text.startswith("Ðáp Ca:") or text.startswith("Đáp Ca:") or text.startswith("Đáp ca:") or "thánh vịnh" in lower_text:
#             matched_section = "psalm"
#         elif text.startswith("Bài Ðọc II:") or text.startswith("Bài Đọc II:") or text.startswith("Bài đọc II:"):
#             matched_section = "reading2"
#         elif text.startswith("Alleluia:") or text.startswith("Tung Hô"):
#             matched_section = "alleluia"
#         elif text.startswith("Phúc Âm:") or text.startswith("Tin Mừng:") or text.startswith("Tin mừng:") or text.startswith("Preaching:"):
#             matched_section = "gospel"

#         if matched_section:
#             save_current_section()
#             current_section = matched_section
#             current_text = []
#             continue

#         if current_section:
#             if not any(noise in lower_text for noise in ["bấm vào đây", "đó là lời chúa", "thứ sáu"]):
#                 if text not in current_text:
#                     current_text.append(text)

#     save_current_section()
#     return viet_dict

# def get_thanhlinh_url_dynamically(date_str):
#     try:
#         target_date = datetime.strptime(date_str, "%m%d%y")
#         anchor_date = datetime(2026, 8, 8)
#         anchor_id = 587
#         delta_days = (target_date - anchor_date).days
#         computed_id = anchor_id + delta_days
#         return f"https://thanhlinh.net/loi-chua-{computed_id}"
#     except Exception as e:
#         print(f"  -> Offset calculation warning: {e}")
#         return "https://thanhlinh.net/loi-chua-587"


# ==========================================
# 3. STRUCTURED PARSERS (Readings, Psalm & Hymns)
# ==========================================

# def parse_reading_to_dict(raw_text):
#     """Separates a reading string into citation and clean body text, removing leading quotes, announcements, and closings."""
#     if not raw_text:
#         return {"citation": "", "text": ""}
        
#     cleaned = raw_text.strip()
#     lines = cleaned.split("\n")
#     citation_val = ""
#     body_text = ""
    
#     # 1. Extract citation if it sits on its own line
#     if len(lines) > 1 and _looks_like_citation(lines[0]):
#         citation_val = lines[0].strip()
#         body_text = " ".join(lines[1:]).strip()
#     else:
#         # Fallback extraction if the citation is squashed into the same line as the text
#         match = re.match(r'^([1-3]?\s*[A-Z][a-zA-Z]+\.?\s+\d+[:\d,\-\sa-zA-Z]+?)(?=[A-Z“"T])', cleaned)
#         if match:
#             citation_candidate = match.group(1).strip()
#             if _looks_like_citation(citation_candidate) or len(citation_candidate) < 30:
#                 citation_val = citation_candidate
#                 body_text = cleaned[len(match.group(1)):].strip()
                
#     if not body_text:
#         body_text = cleaned
        
#     # 2. Clean leading quotes and trailing punctuation (e.g. removes `"Quote". `)
#     body_text = re.sub(r'^\s*["“][^"”]+["”][\s\.,;]*', '', body_text)
    
#     # 3. Clean standard liturgical intro sentences up to their ending period/colon (e.g. `Trích thư... Rôma. `)
#     intro_prefixes = r'(?:Bài trích|Bài đọc|Tin Mừng|Trích sách|Trích thư|Thư của|Phúc Âm|Bài Ðọc|Bài Đọc|Khởi đầu)'
#     intro_pattern = rf'^{intro_prefixes}[^\.:]+[\.:]\s*'
#     body_text = re.sub(intro_pattern, '', body_text, flags=re.IGNORECASE)
    
#     # 4. Clean standard liturgical closings at the very end of the text
#     body_text = re.sub(r'\s*Ðó là lời Chúa\.?\s*$', '', body_text, flags=re.IGNORECASE)
#     body_text = re.sub(r'\s*Đó là lời Chúa\.?\s*$', '', body_text, flags=re.IGNORECASE)
#     body_text = re.sub(r'\s*The word of the Lord\.?\s*$', '', body_text, flags=re.IGNORECASE)

#     return {
#         "citation": citation_val,
#         "text": body_text.strip()
#     }


# General, book-agnostic scripture citation shape:
# optional "cf.", optional leading book-number (1-3), a short ASCII book token (<=7 letters),
# optional period, then chapter/verse digits with optional ":" or "," and verse-letter suffixes/ranges.
# Matches Mt 5:10 / Ga 8:12 / Kh 1:5ab / Tv 94, 8ab / 1 Cor 13:4-7 / cf. Jn 3:16, etc. -
# ANY book, not just a hardcoded list.
# SCRIPTURE_CITATION_RE = re.compile(
#     r'(?:cf\.\s*)?(?:[1-3]\s?)?[A-Z][a-zA-Z]{0,6}\.?\s*\d{1,3}'
#     r'(?:\s*[:,]\s*\d+[a-z]{0,3}(?:\s*[-–]\s*\d+[a-z]{0,3})?)?'
# )

# def parse_alleluia_text(raw_text):
#     """Strips out redundant opening/closing Alleluia framing and any inline scripture
#     citation (from any book, dash or no dash, leading or trailing), returning the clean
#     verse text plus the citation it found."""
#     if not raw_text:
#         return {"citation": "", "text": "Alleluia, alleluia!"}

#     cleaned = raw_text.strip()
#     citation_text = ""

#     # 1. If the citation sits on its own line (e.g. USCCB's separate address element),
#     #    pull it off first.
#     lines = [l.strip() for l in cleaned.split("\n") if l.strip()]
#     if len(lines) > 1 and (_looks_like_citation(lines[0]) or SCRIPTURE_CITATION_RE.fullmatch(lines[0])):
#         citation_text = lines[0].strip()
#         cleaned = "\n".join(lines[1:]).strip()

#     cleaned_flat = " ".join(cleaned.split())

#     # 2. Find & strip ANY inline scripture citation, wherever it sits, whatever the book.
#     #    (Handles no-dash, dash-prefixed, and trailing-citation formats alike.)
#     def _grab(m):
#         nonlocal citation_text
#         if not citation_text:
#             citation_text = m.group(0).strip(" -–—")
#         return " "
#     cleaned_flat = SCRIPTURE_CITATION_RE.sub(_grab, cleaned_flat)

#     # 3. Tidy up leftover dashes/double-spaces left behind where the citation used to sit.
#     cleaned_flat = re.sub(r'\s*[-–—]\s*[-–—]\s*', ' - ', cleaned_flat)
#     cleaned_flat = re.sub(r'\s{2,}', ' ', cleaned_flat).strip()

#     # 4. Strip opening/closing Alleluia framing (English & Vietnamese).
#     verse_core = re.sub(r'^(?:R\.\s*)?Alleluia,?\s*alleluia[!.]?\s*(?:-\s*)?', '', cleaned_flat, flags=re.IGNORECASE).strip()
#     verse_core = re.sub(r'\s*(?:-\s*)?(?:R\.\s*)?Alleluia[!.]?\s*$', '', verse_core, flags=re.IGNORECASE).strip()
#     verse_core = re.sub(r'\s*(?:-\s*)?(?:R\.\s*)?Alleluia,?\s*alleluia[!.]?\s*$', '', verse_core, flags=re.IGNORECASE).strip()
#     verse_core = re.sub(r'\s*R\.\s*Alleluia,?\s*$', '', verse_core, flags=re.IGNORECASE).strip()
#     verse_core = re.sub(r'\s*R\.\s*$', '', verse_core, flags=re.IGNORECASE).strip()
#     verse_core = re.sub(r'^[-–—,\.\s]+', '', verse_core).strip()

#     return {
#         "citation": citation_text,
#         "text": verse_core if verse_core else cleaned_flat
#     }

# def parse_usccb_psalm_to_dict(raw_text):
#     """Specifically parses USCCB English psalm strings into citation, response, and clean stanzas without repeating the response."""
#     if not raw_text:
#         return {"citation": "", "response": "", "verses": {}}
        
#     cleaned = raw_text.strip()
#     citation_text = ""
#     response_text = ""
#     verses_list = []
    
#     lines = [l.strip() for l in cleaned.split("\n") if l.strip()]
#     if lines and (re.search(r'(?:Psalm|Ps\.?)\s*\d+', lines[0], re.IGNORECASE) or len(lines[0]) < 40):
#         citation_text = lines[0].strip()
#         cleaned = "\n".join(lines[1:]).strip()

#     cleaned_flat = " ".join(cleaned.split())

#     if "R." in cleaned_flat:
#         parts = cleaned_flat.split("R.")
#         if len(parts) > 1:
#             refrain_segment = parts[1].strip()
#             match_end = re.search(r'(\(?[0-9a-zA-Z\.\s]+\)?\.\s*["\u201d]?|[\.?!]\s*["\u201d]?)', refrain_segment)
#             if match_end and match_end.start() > 5:
#                 end_idx = match_end.end()
#                 response_text = "R. " + refrain_segment[:end_idx].strip()
#             else:
#                 response_text = "R. " + refrain_segment.split('.')[0] + "."

#     body_text = cleaned_flat
#     if response_text in body_text:
#         body_text = body_text.replace(response_text, "", 1)

#     core_resp_phrase = re.sub(r'^R\.\s*(\([^\)]+\))?\s*', '', response_text).strip()

#     raw_chunks = body_text.split("R.")
#     for chunk in raw_chunks:
#         chunk_clean = chunk.strip()
        
#         if core_resp_phrase:
#             chunk_clean = chunk_clean.replace(core_resp_phrase, "").strip()
            
#         chunk_clean = re.sub(r'^[\.,;\s]+', '', chunk_clean).strip()
#         chunk_clean = re.sub(r'R\.\s*$', '', chunk_clean).strip()
        
#         if chunk_clean and len(chunk_clean) > 5:
#             verses_list.append(chunk_clean)

#     verses_dict = {}
#     for i, verse_text in enumerate(verses_list, start=1):
#         clean_v = verse_text.strip()
#         if not re.match(r'^\d+\.', clean_v):
#             verses_dict[f"verse{i}"] = f"{i}. {clean_v}"
#         else:
#             verses_dict[f"verse{i}"] = clean_v

#     return {
#         "citation": citation_text,
#         "response": response_text,
#         "verses": verses_dict
#     }

# def parse_thanhlinh_psalm_to_dict(raw_text):
#     """Specifically parses ThanhLinh Vietnamese psalm strings into citation, response, and numbered verses."""
#     if not raw_text:
#         return {"citation": "", "response": "", "verses": {}}
        
#     cleaned = raw_text.strip()
#     citation_text = ""
#     response_text = ""
#     verses_list = []
    
#     match_broader = re.search(r'^((?:Tv|Ps|Thánh\s*Vịnh)[\d\s:,\-a-zà-ỹÀ-Ỹ]+)', cleaned, re.IGNORECASE)
#     if match_broader and len(match_broader.group(1)) < 30:
#         citation_text = match_broader.group(1).strip()
#         cleaned = cleaned[len(citation_text):].strip()

#     cleaned_flat = " ".join(cleaned.split())

#     marker = None
#     if "Ðáp:" in cleaned_flat:
#         marker = "Ðáp:"
#     elif "Đáp:" in cleaned_flat:
#         marker = "Đáp:"
#     elif "R." in cleaned_flat:
#         marker = "R."
        
#     if marker:
#         parts = cleaned_flat.split(marker)
#         if len(parts) > 1:
#             refrain_segment = parts[1].strip()
#             match_end = re.search(r'(\(?[0-9a-zA-ZÀ-Ỹa-zà-ỹ\.\s]+\)?\.\s*["\u201d]?|[\.?!]\s*["\u201d]?)', refrain_segment)
#             if match_end and match_end.start() > 5:
#                 end_idx = match_end.end()
#                 response_text = marker + " " + refrain_segment[:end_idx].strip()
#             else:
#                 response_text = marker + " " + refrain_segment.split('.')[0] + "."

#     body_text = cleaned_flat
#     if response_text in body_text:
#         body_text = body_text.replace(response_text, "", 1)

#     split_stanzas = re.split(r'(?=\s+\d+\.\s+)', body_text)
#     for stz in split_stanzas:
#         stz_clean = stz.strip()
#         if re.match(r'^\d+\.\s+', stz_clean):
#             for r_marker in ["R.", "Ðáp:", "Đáp:"]:
#                 if r_marker in stz_clean:
#                     stz_clean = stz_clean.split(r_marker)[0].strip()
#             content_only = re.sub(r'^\d+\.\s*', '', stz_clean).strip()
#             if content_only:
#                 verses_list.append(stz_clean)

#     verses_dict = {}
#     for i, verse_text in enumerate(verses_list, start=1):
#         clean_v = verse_text.strip()
#         if not re.match(r'^\d+\.', clean_v):
#             verses_dict[f"verse{i}"] = f"{i}. {clean_v}"
#         else:
#             verses_dict[f"verse{i}"] = clean_v

#     return {
#         "citation": citation_text,
#         "response": response_text,
#         "verses": verses_dict
#     }

# def prepare_template_data(user_inputs, scraped_eng, scraped_viet):
#     """Populates the master dictionary using tailored parsers for English and Vietnamese with validation warnings."""
#     final_data = {
#         "feast_day": scraped_eng.get("feast_day"),
#         "citations": {},
#         "eng": {}, 
#         "viet": {},
#         "hymns": user_inputs.get("hymns", {}),
#         "warnings": []
#     }
    
#     required_keys = ["feast_day", "reading1", "psalm", "alleluia", "gospel"]
    
#     sections = ["reading1", "psalm", "reading2", "alleluia", "gospel"]
    
#     for section in sections:
#         idx = user_inputs.get(section, {}).get("option_index", 0)
        
#         # 1. Parse English
#         eng_parsed = None
#         if section in scraped_eng and len(scraped_eng[section]) > idx:
#             val_eng = scraped_eng[section][idx]
#             if section == "psalm" and val_eng:
#                 eng_parsed = parse_usccb_psalm_to_dict(val_eng)
#             elif section == "alleluia" and val_eng:
#                 eng_parsed = parse_alleluia_text(val_eng)
#             elif val_eng:
#                 eng_parsed = parse_reading_to_dict(val_eng)
                
#         # 2. Parse Vietnamese
#         viet_parsed = None
#         if section in scraped_viet and len(scraped_viet[section]) > idx:
#             val_viet = scraped_viet[section][idx]
#             if section == "psalm" and val_viet:
#                 viet_parsed = parse_thanhlinh_psalm_to_dict(val_viet)
#             elif section == "alleluia" and val_viet:
#                 viet_parsed = parse_alleluia_text(val_viet)
#             elif val_viet:
#                 viet_parsed = parse_reading_to_dict(val_viet)

#         # 3. Store top-level shared citation
#         citation_val = ""
#         if eng_parsed and eng_parsed.get("citation"):
#             citation_val = eng_parsed.get("citation")
#         elif viet_parsed and viet_parsed.get("citation"):
#             citation_val = viet_parsed.get("citation")
            
#         final_data["citations"][section] = citation_val

#         # 4. Assign text bodies
#         if eng_parsed:
#             if section == "psalm":
#                 final_data["eng"]["psalm"] = {
#                     "response": eng_parsed.get("response"),
#                     "verses": eng_parsed.get("verses")
#                 }
#             elif section == "alleluia":
#                 final_data["eng"]["alleluia"] = eng_parsed.get("text")
#             else:
#                 final_data["eng"][section] = eng_parsed.get("text")
#         else:
#             final_data["eng"][section] = None

#         if viet_parsed:
#             if section == "psalm":
#                 final_data["viet"]["psalm"] = {
#                     "response": viet_parsed.get("response"),
#                     "verses": viet_parsed.get("verses")
#                 }
#             elif section == "alleluia":
#                 final_data["viet"]["alleluia"] = viet_parsed.get("text")
#             else:
#                 final_data["viet"][section] = viet_parsed.get("text")
#         else:
#             final_data["viet"][section] = None

    
#     # 1. Check for missing or empty mandatory keys across eng/viet/top-level
#     for key in required_keys:
#         if key == "feast_day":
#             if not final_data.get("feast_day") or final_data.get("feast_day") == "Daily Readings":
#                 final_data["warnings"].append("Warning: 'feast_day' was not properly identified.")
#         else:
#             eng_val = final_data["eng"].get(key)
#             viet_val = final_data["viet"].get(key)
#             if not eng_val and not viet_val:
#                 final_data["warnings"].append(f"Warning: Required section '{key}' is missing text in both English and Vietnamese.")

#     # 2. Check for missing citations for any section that has text
#     all_sections = ["reading1", "psalm", "reading2", "alleluia", "gospel"]
#     for key in all_sections:
#         has_text = final_data["eng"].get(key) or final_data["viet"].get(key)
#         # For psalm, text is a dict with responses/verses, so check accordingly or check presence
#         if key == "psalm":
#             has_text = final_data["eng"].get("psalm") or final_data["viet"].get("psalm")
            
#         if has_text and not final_data["citations"].get(key):
#             final_data["warnings"].append(f"Warning: Citation for section '{key}' is missing.")

#     # 3. Check if it's a Sunday and any reading is completely missing text
#     date_str = user_inputs.get("date")
#     if date_str:
#         try:
#             dt = datetime.strptime(date_str, "%m%d%y")
#             is_sunday = (dt.weekday() == 6)
#             if is_sunday:
#                 for key in all_sections:
#                     if not final_data["eng"].get(key) and not final_data["viet"].get(key):
#                         final_data["warnings"].append(f"Sunday Warning: No text found for '{key}' on a Sunday service.")
#         except Exception:
#             pass
        
#     return final_data


# ==========================================
# 4. 4-PANEL BOOKLET GENERATION (Imposition)
# ==========================================

# def set_col_spacing(section, num_cols=2, space_in_inches=1.2):
#     sectPr = section._sectPr
#     cols = sectPr.xpath('./w:cols')
#     if cols:
#         cols[0].set(qn('w:num'), str(num_cols))
#         cols[0].set(qn('w:space'), str(int(space_in_inches * 1440)))

# def add_heading_with_citation(doc, label, citation):
#     """Creates a heading line with the label on the left and citation pushed to the far right margin."""
#     p = doc.add_paragraph()
#     p.paragraph_format.space_before = Pt(4)
#     p.paragraph_format.space_after = Pt(2)
    
#     p.paragraph_format.tab_stops.add_tab_stop(Inches(4.4), WD_TAB_ALIGNMENT.RIGHT)
    
#     r_label = p.add_run(label)
#     r_label.font.bold = True
#     r_label.font.size = Pt(10.5)
#     r_label.font.color.rgb = RGBColor(0, 0, 0)
    
#     if citation:
#         r_tab = p.add_run("\t")
#         r_cite = p.add_run(citation)
#         r_cite.font.bold = False
#         r_cite.font.italic = False
#         r_cite.font.size = Pt(9.5)
#         r_cite.font.color.rgb = RGBColor(80, 80, 80)

# def add_reading_section(doc, label, citation, text):
#     """Renders heading with right-aligned citation and body text into the document."""
#     if not text:
#         return
        
#     add_heading_with_citation(doc, label, citation)
        
#     p_txt = doc.add_paragraph()
#     p_txt.paragraph_format.space_after = Pt(3)
#     r_txt = p_txt.add_run(text)
#     r_txt.font.size = Pt(9.5)

# def add_psalm_section(doc, final_data, lang_choice):
#     citations = final_data.get("citations", {})
#     psalm_data = final_data["viet"].get("psalm") if lang_choice == "viet" else final_data["eng"].get("psalm")
    
#     if not psalm_data:
#         return
        
#     add_heading_with_citation(doc, "RESPONSORIAL PSALM / ĐÁP CA", citations.get("psalm", ""))
    
#     response_text = psalm_data.get("response", "")
#     if response_text:
#         p_resp = doc.add_paragraph()
#         p_resp.paragraph_format.space_after = Pt(4)
#         r_run = p_resp.add_run(response_text)
#         r_run.font.bold = True
#         r_run.font.size = Pt(9.5)
        
#     verses_dict = psalm_data.get("verses", {})
#     for v_key, verse_text in verses_dict.items():
#         p_v = doc.add_paragraph()
#         p_v.paragraph_format.space_after = Pt(4)
#         p_v.paragraph_format.space_before = Pt(0)
#         p_v.paragraph_format.left_indent = Inches(0.15)
        
#         v_run = p_v.add_run(verse_text + " ")
#         v_run.font.size = Pt(9.5)
        
#         r_run = p_v.add_run("R.")
#         r_run.font.bold = True
#         r_run.font.size = Pt(9.5)

# def add_hymn_section(doc, label, hymn_data):
#     """Renders a hymn using either structured (chorus/verses) or vanilla text format."""
#     if not hymn_data or not hymn_data.get("title"):
#         return
        
#     add_heading_with_citation(doc, label.upper(), "")
    
#     p_title = doc.add_paragraph()
#     p_title.paragraph_format.space_after = Pt(2)
#     r_title = p_title.add_run(f"Thánh Ca: {hymn_data.get('title')}")
#     r_title.font.bold = True
#     r_title.font.size = Pt(9.5)
    
#     if "text" in hymn_data:
#         p_txt = doc.add_paragraph()
#         p_txt.paragraph_format.space_after = Pt(4)
#         r_txt = p_txt.add_run(hymn_data.get("text"))
#         r_txt.font.size = Pt(9.5)
#     else:
#         chorus = hymn_data.get("chorus", "")
#         if chorus:
#             p_ch = doc.add_paragraph()
#             p_ch.paragraph_format.space_after = Pt(4)
#             r_ch = p_ch.add_run(chorus)
#             r_ch.font.bold = True
#             r_ch.font.size = Pt(9.5)
            
#         verses = hymn_data.get("verses", {})
#         for v_key, v_text in verses.items():
#             p_v = doc.add_paragraph()
#             p_v.paragraph_format.space_after = Pt(4)
#             p_v.paragraph_format.space_before = Pt(0)
#             p_v.paragraph_format.left_indent = Inches(0.15)
            
#             r_v = p_v.add_run(v_text)
#             r_v.font.size = Pt(9.5)
        
#     doc.add_paragraph()

# def create_booklet_docx(user_inputs, final_data, filename="Mass_Booklet_Imposed.docx"):
#     doc = Document()
    
#     section = doc.sections[0]
#     section.page_width = Inches(11.0)
#     section.page_height = Inches(8.5)
#     section.top_margin = Inches(0.5)
#     section.bottom_margin = Inches(0.5)
#     section.left_margin = Inches(0.5)
#     section.right_margin = Inches(0.5)
#     set_col_spacing(section, num_cols=2, space_in_inches=1.2)
    
#     hymns = final_data.get("hymns", {})
#     citations = final_data.get("citations", {})
    
#     # --- PANEL 4: BACK COVER (Left side of Page 1) ---
#     p_back_title = doc.add_paragraph()
#     r_bt = p_back_title.add_run("CÂU NGUYỆN & KẾT LỄ\n")
#     r_bt.font.bold = True
#     r_bt.font.size = Pt(11)
    
#     p_body = doc.add_paragraph()
#     p_body.paragraph_format.space_after = Pt(3)
#     r_body = p_body.add_run(
#         "Anima Christi:\n"
#         "Soul of Christ, sanctify me.\n"
#         "Body of Christ, save me.\n"
#         "Blood of Christ, embolden me.\n"
#         "Water from the side of Christ, wash me.\n"
#         "Passion of Christ, strengthen me.\n"
#         "O good Jesus, hear me.\n"
#         "Within your wounds hide me.\n"
#         "Never permit me to be parted from you.\n"
#         "From the evil Enemy defend me.\n"
#         "At the hour of my death call me and bid me come to you,\n"
#         "that with your Saints I may praise you for age upon age.\n"
#         "Amen.\n"
#     )
#     r_body.font.size = Pt(9.5)
    
#     if "communion" in hymns:
#         add_hymn_section(doc, "Communion Hymn / Thánh Ca Hiệp Lễ", hymns["communion"])
        
#     if "recessional" in hymns:
#         add_hymn_section(doc, "Recessional Hymn / Thánh Ca Kết Lễ", hymns["recessional"])
        
#     doc.add_paragraph()
    
#     p_break1 = doc.add_paragraph()
#     p_break1.add_run().add_break(docx.enum.text.WD_BREAK.COLUMN)
    
#     # --- PANEL 1: FRONT COVER (Right side of Page 1) ---
#     p_front = doc.add_paragraph()
#     p_front.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
#     org_text = user_inputs.get("organization", "ORGANIZATION")
#     r_org = p_front.add_run(f"{org_text}\n")
#     r_org.font.size = Pt(12)
#     r_org.font.bold = True
#     r_org.font.color.rgb = RGBColor(100, 100, 100)
    
#     event_text = user_inputs.get("event_name", "Mass Booklet")
#     r_event = p_front.add_run(f"{event_text}\n")
#     r_event.font.size = Pt(14)
#     r_event.font.bold = True
    
#     r_feast = p_front.add_run(f"{final_data.get('feast_day', 'Mass Readings')}\n")
#     r_feast.font.size = Pt(11.5)
    
#     formatted_date = datetime.strptime(user_inputs.get("date"), "%m%d%y").strftime("%B %d, %Y")
#     r_date = p_front.add_run(f"{formatted_date}\n\n")
#     r_date.font.size = Pt(10)
#     r_date.font.italic = True
    
#     if "opening" in hymns:
#         add_hymn_section(doc, "Opening Hymn / Thánh Ca Nhập Lễ", hymns["opening"])
    
#     p_break2 = doc.add_paragraph()
#     p_break2.add_run().add_break(docx.enum.text.WD_BREAK.PAGE)
    
#     # --- PANEL 2: INSIDE LEFT (Page 2 of booklet) ---
#     sections_left = [
#         ("reading1", "READING 1 / BÀI ĐỌC I"),
#         ("psalm", "RESPONSORIAL PSALM / ĐÁP CA"),
#         ("reading2", "READING 2 / BÀI ĐỌC II")
#     ]
    
#     for key, label in sections_left:
#         conf = user_inputs.get(key, {"lang": "eng"})
#         lang = conf.get("lang", "eng")
        
#         if key == "psalm":
#             add_psalm_section(doc, final_data, lang)
#         else:
#             text_body = final_data["viet"].get(key) if lang == "viet" else final_data["eng"].get(key)
#             if text_body:
#                 add_reading_section(doc, label, citations.get(key, ""), text_body)
#                 doc.add_paragraph()
            
#     p_break3 = doc.add_paragraph()
#     p_break3.add_run().add_break(docx.enum.text.WD_BREAK.COLUMN)
    
#     # --- PANEL 3: INSIDE RIGHT (Page 3 of booklet) ---
#     sections_right = [
#         ("alleluia", "ALLELUIA"),
#         ("gospel", "GOSPEL / TIN MỪNG")
#     ]
    
#     for key, label in sections_right:
#         conf = user_inputs.get(key, {"lang": "eng"})
#         lang = conf.get("lang", "eng")
#         text_body = final_data["viet"].get(key) if lang == "viet" else final_data["eng"].get(key)
#         if text_body:
#             display_label = label
#             if key == "alleluia":
#                 eng_all_text = final_data["eng"].get("alleluia", "") or ""
#                 if "alleluia" not in eng_all_text.lower():
#                     display_label = "GOSPEL ACCLAMATION / TUNG HÔ TIN MỪNG"
                    
#             add_reading_section(doc, display_label, citations.get(key, ""), text_body)
#             doc.add_paragraph()
            
#     if "offertory" in hymns:
#         add_hymn_section(doc, "Offertory Hymn / Thánh Ca Tiến Lễ", hymns["offertory"])

#     doc.save(filename)
#     print(f"Booklet handout successfully saved as {filename}!")


: 

In [2]:
# ==========================================
# 5. MAIN EXECUTION
# ==========================================

async def main():
    target_date = "080726"  
    
    print(f"Resolving ThanhLinh URL dynamically for date: {target_date}...")
    resolved_viet_url = get_thanhlinh_url_dynamically(target_date)
    print(f"  -> Discovered ThanhLinh URL: {resolved_viet_url}")

    app_inputs = {
        "date": target_date,  
        "viet_url": resolved_viet_url,
        "organization": "ĐOÀN ANÊ THÀNH",
        "event_name": "Hike & Mass at Discovery Park",
        "hymns": {
            "opening": {
                "title": "Khúc Ca Mở Lễ",
                "chorus": "ĐK. Xin hồng ân Chúa tràn trề trên đoàn con bước vào nhà Chúa...",
                "verses": {
                    "verse1": "1. Đoàn con tiến bước về đền thánh Ngài, hòa tiếng hát dâng ngàn lời ca...",
                    "verse2": "2. Nguyện xin dâng tiến muôn vàn tâm tư, cùng năm tháng sống đượm tình mến yêu..."
                }
            },
            "offertory": {
                "title": "Tiến Lễ Tin Yêu",
                "chorus": "ĐK. Con xin dâng lên bánh miến với rượu nho nồng...",
                "verses": {
                    "verse1": "1. Từ muôn phương chúng con về đây dâng hiến..."
                }
            },
            "communion": {
                "title": "Con Thờ Lạy Chúa",
                "chorus": "ĐK. Chúa ơi con tin có Chúa ngự nơi đây...",
                "verses": {
                    "verse1": "1. Dù ngàn gian khó con không sợ gì vì Chúa ở cùng con..."
                }
            },
            "recessional": {
                "title": "Salve Regina",
                "text": "Salve, Regina, mater misericordiae, vita, dulcedo, et spes nostra, salve.\nAd te clamamus, exsules filii Hevae.\nEia, ergo, advocata nostra, illos tuos misericordes oculos ad nos converte."
            }
        },
        "reading1": {"lang": "eng", "option_index": 0},
        "psalm":    {"lang": "eng", "option_index": 0},
        "reading2": {"lang": "eng", "option_index": 0},
        "alleluia": {"lang": "eng", "option_index": 0},
        "gospel":   {"lang": "eng", "option_index": 0}    
    }
    
    print("Scraping USCCB & ThanhLinh...")
    eng_readings = await scrape_usccb_async(app_inputs["date"])
    viet_readings = scrape_thanhlinh(app_inputs["viet_url"])
    
    final_document_data = prepare_template_data(app_inputs, eng_readings, viet_readings)
    
    print("\n================ MASTER DICTIONARY OUTPUT ================")
    print(json.dumps(final_document_data, indent=4, ensure_ascii=False))
    print("=========================================================\n")
    
    print("\nGenerating Imposed Booklet Word Document...")
    create_booklet_docx(app_inputs, final_document_data, "Mass_Booklet_Imposed.docx")

await main() 

Resolving ThanhLinh URL dynamically for date: 080726...
  -> Discovered ThanhLinh URL: https://thanhlinh.net/loi-chua-586
Scraping USCCB & ThanhLinh...
  -> Navigating to USCCB for 080726...

================ MASTER DICTIONARY OUTPUT ================
{
    "feast_day": "Friday of the Eighteenth Week in Ordinary Time",
    "citations": {
        "reading1": "",
        "psalm": "Deuteronomy 32:35cd-36ab, 39abcd, 41",
        "reading2": "",
        "alleluia": "Matthew 5:10",
        "gospel": "Matthew 16:24-28"
    },
    "eng": {
        "reading1": "Nahum 2:1, 3; 3:1-3, 6-7\nSee, upon the mountains there advances the bearer of good news, announcing peace! Celebrate your feasts, O Judah, fulfill your vows! For nevermore shall you be invaded by the scoundrel; he is completely destroyed. The LORD will restore the vine of Jacob, the pride of Israel, Though ravagers have ravaged them and ruined the tendrils.Woe to the bloody city, all lies, full of plunder, whose looting never stops! The 

# Notes

1. August 7, 2026 has no Reading 1 citation, but the citation is in the reading. The Vietnamese has no alleluia.
2. If all null for Viet, say X page is not yet found.
3. Add note for optional readings available (not supported yet)
4. Add warnings: If something was not found or (2) If Sunday and no secnd readings was found.
Now, feast_day, reading1, psalm, alleluia, gospel will always be there. I think it would be a good idea if we had another thing in our final_document_data that had warnings if one of these weer not found or if on a Sunday, no readings was found.
Also, I would prefer if we changed the "alleluia" in the final_document_data dictionary to "acclamation"